# LeetCode 2289 — Steps to Make Array Non-decreasing bằng Python

Link đề: https://leetcode.com/problems/steps-to-make-array-non-decreasing/ · Độ khó: **Medium** · Chủ đề: Array, Linked List, Stack, Monotonic Stack

Skeleton LeetCode:

```python
class Solution:
    def totalSteps(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô phỏng chính xác một quá trình **đồng thời** (mọi phần tử bị xóa trong cùng một bước được quyết định theo trạng thái **đầu bước**), và thấy vì sao mô phỏng thô là O(n²).
2. Chuyển từ "đếm số bước" sang "tính **thời điểm bị xóa** của từng phần tử": đáp án = `max` thời điểm bị xóa.
3. Chứng minh công thức then chốt: phần tử `i` bị xóa ở bước `1 + max(thời điểm bị xóa của các phần tử nằm giữa nó và **phần tử lớn hơn gần nhất bên trái**)`, và không bao giờ bị xóa nếu không có phần tử lớn hơn bên trái.
4. Tính công thức đó trong O(n) bằng **ngăn xếp đơn điệu** (duyệt trái → phải, hoặc phải → trái như doocs).
5. Một cách O(n) khác: mô phỏng bằng **danh sách liên kết** chỉ xét lại những "ứng viên" có hàng xóm trái vừa thay đổi.

## 2. Đề bài & ví dụ

Cho mảng `nums` (chỉ số từ 0). Trong **một bước**, xóa **đồng thời** mọi phần tử `nums[i]` mà `nums[i − 1] > nums[i]` (với `0 < i < len(nums)`). Trả về số bước được thực hiện cho tới khi `nums` trở thành mảng **không giảm**.

**Ví dụ 1:** `nums = [5,3,4,4,7,3,6,11,8,5,11]` → `3`.
- Bước 1: `[5,3,4,4,7,3,6,11,8,5,11]` → `[5,4,4,7,6,11,11]`
- Bước 2: `[5,4,4,7,6,11,11]` → `[5,4,7,11,11]`
- Bước 3: `[5,4,7,11,11]` → `[5,7,11,11]`

**Ví dụ 2:** `nums = [4,5,7,7,13]` → `0` (đã không giảm).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `1 ≤ nums[i] ≤ 10⁹`.

In [1]:
from typing import List
import random, time

examples = [([5, 3, 4, 4, 7, 3, 6, 11, 8, 5, 11], 3), ([4, 5, 7, 7, 13], 0)]
for a, exp in examples:
    print(a, "-> kỳ vọng", exp)

[5, 3, 4, 4, 7, 3, 6, 11, 8, 5, 11] -> kỳ vọng 3
[4, 5, 7, 7, 13] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Cập nhật **đồng thời** khác cập nhật **tuần tự**

Trong một bước, điều kiện `nums[i−1] > nums[i]` được xét trên mảng **trước** bước đó. Nếu xóa ngay khi duyệt, phần tử kế tiếp sẽ bị so với một hàng xóm "mới" — sai.

In [2]:
def one_step(a):
    return [a[0]] + [a[i] for i in range(1, len(a)) if not (a[i - 1] > a[i])]

a = [5, 3, 4]
print("đồng thời:", one_step(a))
seq = [a[0]]
for x in a[1:]:
    if not (seq[-1] > x): seq.append(x)        # tuần tự: so với phần tử còn lại cuối cùng
print("tuần tự  :", seq, "(sai: 4 bị so với 5 ngay trong bước 1)")

đồng thời: [5, 4]
tuần tự  : [5] (sai: 4 bị so với 5 ngay trong bước 1)


### 3.2 Phần tử lớn hơn gần nhất bên trái (previous greater element)

`pg[i]` = chỉ số `j < i` lớn nhất có `nums[j] > nums[i]` (hoặc `−1`). Tính O(n) bằng ngăn xếp đơn điệu giảm.

In [3]:
def previous_greater(a):
    pg = [-1] * len(a); st = []
    for i, x in enumerate(a):
        while st and a[st[-1]] <= x: st.pop()
        pg[i] = st[-1] if st else -1
        st.append(i)
    return pg

a = examples[0][0]
print(list(enumerate(a))); print("pg:", previous_greater(a))

[(0, 5), (1, 3), (2, 4), (3, 4), (4, 7), (5, 3), (6, 6), (7, 11), (8, 8), (9, 5), (10, 11)]
pg: [-1, 0, 0, 0, -1, 4, 4, -1, 7, 8, -1]


### 3.3 Phần tử không có ai lớn hơn bên trái thì **không bao giờ** bị xóa

Hàng xóm trái của nó luôn là một phần tử ≤ nó (hoặc không có). Các phần tử này chính là **tiền tố cực đại** (prefix maxima, tính cả bằng nhau) — chúng tạo thành mảng cuối cùng.

In [4]:
def prefix_maxima(a):
    out = []; best = 0
    for x in a:
        if x >= best: out.append(x); best = x
    return out
print(prefix_maxima(examples[0][0]), "= mảng cuối của ví dụ 1")

[5, 7, 11, 11] = mảng cuối của ví dụ 1


### 3.4 Danh sách liên kết đôi trên mảng (`prev`, `nxt`)

Xóa một phần tử khỏi dãy trong O(1): nối `prev` và `nxt` của nó với nhau.

In [5]:
vals = [10, 20, 30, 40]; prv = [-1, 0, 1, 2]; nxt = [1, 2, 3, -1]
def unlink(i):
    if prv[i] != -1: nxt[prv[i]] = nxt[i]
    if nxt[i] != -1: prv[nxt[i]] = prv[i]
unlink(1); unlink(2)
i, out = 0, []
while i != -1: out.append(vals[i]); i = nxt[i]
print(out)

[10, 40]


## 4. Ý tưởng nền tảng / trực giác

Gọi `T[i]` = **bước mà phần tử `i` bị xóa** (0 nếu không bao giờ). Số bước cần = `max T[i]`, vì sau bước `max T` không còn ai bị xóa nữa và mảng còn lại (các tiền tố cực đại) là không giảm.

**Ai xóa `i`?** Xét `j = pg[i]` (lớn hơn gần nhất bên trái). Mọi phần tử nằm giữa `j` và `i` đều ≤ `nums[i]`, nên **chưa** thể xóa `i` khi một trong chúng còn là hàng xóm trái của `i`. Khi tất cả chúng đã bị xóa (ở bước `max T[k]`), hàng xóm trái của `i` là một phần tử **> `nums[i]`** (là `j`, hoặc nếu `j` cũng đã bị xóa thì là "kẻ xóa `j`", còn lớn hơn nữa) ⇒ `i` bị xóa ngay bước tiếp theo:

`T[i] = 1 + max(T[k] : j < k < i)` (max của tập rỗng = 0), và `T[i] = 0` nếu không có `j`.

Tính tay ví dụ 1 `[5,3,4,4,7,3,6,11,8,5,11]`:

| i | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|---|
| nums | 5 | 3 | 4 | 4 | 7 | 3 | 6 | 11 | 8 | 5 | 11 |
| pg | − | 0 | 0 | 0 | − | 4 | 4 | − | 7 | 8 | − |
| giữa pg và i | | ∅ | {1} | {1,2} | | ∅ | {5} | | ∅ | ∅ | |
| T | 0 | 1 | 2 | 3 | 0 | 1 | 2 | 0 | 1 | 1 | 0 |

`max T = 3` ✔. Phần tử 4 ở chỉ số 3 phải chờ `3` (bước 1) và `4` ở chỉ số 2 (bước 2) bị xóa rồi mới "lộ" ra trước `5`.

In [6]:
def death_times_quadratic(a):
    pg = previous_greater(a); T = [0] * len(a)
    for i in range(len(a)):
        if pg[i] != -1:
            T[i] = 1 + max((T[k] for k in range(pg[i] + 1, i)), default=0)
    return T

print(death_times_quadratic(examples[0][0]))

[0, 1, 2, 3, 0, 1, 2, 0, 1, 1, 0]


## 5. Cách 1 — Brute Force / baseline

### 5.1 Mô phỏng từng bước — O(n · số bước) = O(n²) worst

In [7]:
def brute(nums):
    a = list(nums); steps = 0
    while True:
        b = one_step(a)
        if len(b) == len(a): return steps
        a = b; steps += 1

for a, exp in examples:
    print(brute(a), "OK" if brute(a) == exp else "SAI")

3 OK
0 OK


**Worst case:** `[n, 1, 2, 3, …, n−1]` — mỗi bước chỉ xóa **một** phần tử (phần tử ngay sau `n`), nên cần `n − 1` bước, mỗi bước O(n) ⇒ O(n²) ≈ `10¹⁰` với `n = 10⁵`.

In [8]:
for n in [500, 1000, 2000]:
    a = [n] + list(range(1, n))
    t0 = time.perf_counter(); s = brute(a); dt = time.perf_counter() - t0
    print(f"n={n:<5} bước={s:<5} {dt*1000:7.1f} ms")

n=500   bước=499       6.2 ms
n=1000  bước=999      27.5 ms
n=2000  bước=1999     98.5 ms


### 5.2 Baseline 2 — công thức `T[i]` tính trực tiếp — O(n²) worst

Cách ở mục 4 (`death_times_quadratic`) đúng nhưng việc lấy `max` trên đoạn giữa là O(n) mỗi phần tử. **Nút thắt:** các đoạn "giữa `pg[i]` và `i`" lồng nhau; ngăn xếp đơn điệu cho phép gộp `max` của đoạn giữa khi pop — mỗi phần tử chỉ được xét lại một lần.

## 6. Ý tưởng thuật toán / bước đột phá

Duyệt **trái → phải**, ngăn xếp lưu cặp `(giá trị, T)` với giá trị **giảm ngặt** từ đáy lên đỉnh.

Khi đến `x = nums[i]`:
1. `t = 0`. **Trong khi** đỉnh có giá trị `≤ x`: `t = max(t, T_đỉnh)`, pop. (Những phần tử này nằm giữa `pg[i]` và `i`; thứ còn lại trên đỉnh sau vòng lặp chính là `pg[i]`.)
2. **Nếu** ngăn xếp rỗng (không có ai lớn hơn bên trái) **thì** `T[i] = 0`; **ngược lại** `T[i] = t + 1`.
3. Đẩy `(x, T[i])`; cập nhật đáp án.

**Vì sao `max` qua các phần tử bị pop là đủ?** Một phần tử `k` nằm giữa `pg[i]` và `i` mà đã bị pop **trước đó** (bởi một `k'` với `k < k' < i`, `nums[k'] ≥ nums[k]`) thì `T[k'] ≥ T[k] + 1 > T[k]` (vì `k'` có `pg` — chính phần tử còn lại trong ngăn xếp — nên `T[k'] = 1 + max(…, T[k], …)`). Vậy `max` qua những phần tử **còn** trên ngăn xếp đã bao trùm những phần tử bị pop trước.

**Khấu hao:** mỗi chỉ số push 1 lần, pop ≤ 1 lần ⇒ **O(n)**.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Ngăn xếp đơn điệu trái → phải

In [9]:
def stack_left(nums):
    st = []; ans = 0                       # st: (giá trị, T), giá trị giảm ngặt từ đáy lên
    for x in nums:
        t = 0
        while st and st[-1][0] <= x:
            t = max(t, st.pop()[1])
        t = t + 1 if st else 0
        ans = max(ans, t)
        st.append((x, t))
    return ans

for a, exp in examples:
    print(stack_left(a), "OK" if stack_left(a) == exp else "SAI")

3 OK
0 OK


### 7.2 Cách 3 — Ngăn xếp phải → trái (cách doocs): "mỗi phần tử ăn dần các phần tử bên phải"

Duyệt từ phải. `dp[i]` = số bước để `i` "ăn" hết chuỗi phần tử nhỏ hơn nó ở bên phải (chuỗi mà nó là `pg`). Khi `nums[i] > nums[top]`: `i` sẽ ăn `top`; thời gian `dp[i] = max(dp[i] + 1, dp[top])` — ăn thêm một phần tử tốn 1 bước, nhưng phải chờ `top` ăn xong phần của nó nếu lâu hơn.

In [10]:
def stack_right(nums):
    n = len(nums); dp = [0] * n; st = []
    for i in range(n - 1, -1, -1):
        while st and nums[i] > nums[st[-1]]:
            dp[i] = max(dp[i] + 1, dp[st.pop()])
        st.append(i)
    return max(dp)

for a, exp in examples:
    print(stack_right(a), "OK" if stack_right(a) == exp else "SAI")

3 OK
0 OK


Ở đây `dp[i]` là **thời điểm `i` xóa xong** phần tử cuối cùng mà nó xóa — bằng `max T[k]` trên các `k` có `pg[k] = i` (nếu có) — nên `max dp = max T`.

### 7.3 Cách 4 — Mô phỏng thông minh bằng danh sách liên kết + tập ứng viên — O(n)

Ý: sau bước `r`, chỉ những phần tử có **hàng xóm trái vừa thay đổi** mới có thể bị xóa ở bước `r + 1`. Mỗi phần tử bị xóa sinh ra tối đa một ứng viên mới (phần tử còn sống ngay sau nó) ⇒ tổng số ứng viên O(n).

In [11]:
def linked_simulation(nums):
    n = len(nums)
    prv = list(range(-1, n - 1)); nxt = list(range(1, n + 1)); nxt[-1] = -1
    alive = [True] * n
    cand = [i for i in range(1, n) if nums[i - 1] > nums[i]]
    steps = 0
    while cand:
        # quyết định theo trạng thái đầu bước
        kill = [i for i in cand if alive[i] and prv[i] != -1 and nums[prv[i]] > nums[i]]
        kill = sorted(set(kill))
        if not kill: break
        steps += 1
        for i in kill: alive[i] = False
        lefts = []
        for i in kill:                                  # duyệt tăng dần: prv[i] luôn là phần tử CÒN SỐNG
            p, q = prv[i], nxt[i]
            if p != -1: nxt[p] = q
            if q != -1: prv[q] = p
            lefts.append(p)
        new = []
        for p in set(lefts):                            # ứng viên = phần tử sống ngay sau hàng xóm trái sống
            q = nxt[p] if p != -1 else -1
            if q != -1: new.append(q)
        cand = new
    return steps

for a, exp in examples:
    print(linked_simulation(a), "OK" if linked_simulation(a) == exp else "SAI")

3 OK
0 OK


Ghi chú cài đặt: khi unlink các phần tử bị xóa theo thứ tự chỉ số tăng, `prv[i]` tại thời điểm unlink luôn là phần tử **còn sống** (các phần tử chết bên trái đã được nối qua). Sau khi unlink xong, `nxt[p]` của các hàng xóm sống đó chính là ứng viên cho bước sau. **Đừng** đi theo `nxt[i]` của phần tử đã chết rồi nhảy qua các phần tử chết: với mảng giảm dần, cả dãy dài bị xóa cùng lúc và mỗi phần tử chết lại đi qua toàn bộ phần còn lại ⇒ O(n²) (bản đầu tiên của notebook này đã mắc đúng lỗi đó). Cách này dễ sai hơn cách ngăn xếp nhưng gần với "mô phỏng" nên dễ tin; nó cũng trả được **mảng cuối** và **danh sách bị xóa ở từng bước** nếu cần.

## 8. TRACE từng bước

### 8.1 Trace mô phỏng ví dụ 1

In [12]:
a = examples[0][0]; step = 0
print("bước 0:", a)
while True:
    b = one_step(a)
    if len(b) == len(a): break
    step += 1
    removed = [a[i] for i in range(1, len(a)) if a[i - 1] > a[i]]
    print(f"bước {step}: xóa {removed} -> {b}")
    a = b

bước 0: [5, 3, 4, 4, 7, 3, 6, 11, 8, 5, 11]
bước 1: xóa [3, 3, 8, 5] -> [5, 4, 4, 7, 6, 11, 11]
bước 2: xóa [4, 6] -> [5, 4, 7, 11, 11]
bước 3: xóa [4] -> [5, 7, 11, 11]


### 8.2 Trace ngăn xếp trái → phải ví dụ 1

In [13]:
def trace_left(nums):
    st = []; ans = 0
    for i, x in enumerate(nums):
        t = 0; popped = []
        while st and st[-1][0] <= x:
            v = st.pop(); popped.append(v); t = max(t, v[1])
        T = t + 1 if st else 0
        ans = max(ans, T)
        st.append((x, T))
        print(f"i={i:<2} x={x:<3} pop {popped!s:<22} pg={'-' if len(st) == 1 else st[-2][0]!s:<3} T={T}  st={st}")
    return ans

print("đáp án:", trace_left(examples[0][0]))

i=0  x=5   pop []                     pg=-   T=0  st=[(5, 0)]
i=1  x=3   pop []                     pg=5   T=1  st=[(5, 0), (3, 1)]
i=2  x=4   pop [(3, 1)]               pg=5   T=2  st=[(5, 0), (4, 2)]
i=3  x=4   pop [(4, 2)]               pg=5   T=3  st=[(5, 0), (4, 3)]
i=4  x=7   pop [(4, 3), (5, 0)]       pg=-   T=0  st=[(7, 0)]
i=5  x=3   pop []                     pg=7   T=1  st=[(7, 0), (3, 1)]
i=6  x=6   pop [(3, 1)]               pg=7   T=2  st=[(7, 0), (6, 2)]
i=7  x=11  pop [(6, 2), (7, 0)]       pg=-   T=0  st=[(11, 0)]
i=8  x=8   pop []                     pg=11  T=1  st=[(11, 0), (8, 1)]
i=9  x=5   pop []                     pg=8   T=1  st=[(11, 0), (8, 1), (5, 1)]
i=10 x=11  pop [(5, 1), (8, 1), (11, 0)] pg=-   T=0  st=[(11, 0)]
đáp án: 3


Đọc dòng `i=3`: `x = 4` pop `(4, 2)` (phần tử ở chỉ số 2, cũng có giá trị 4 — **bằng** nên pop) ⇒ `t = 2`; còn `(5, 0)` ⇒ `T = 3`. Phần tử này bị xóa ở bước 3, đúng như mô phỏng (bước 3 xóa số `4`).

### 8.3 Trace ngăn xếp phải → trái (doocs) trên ví dụ nhỏ `[10, 1, 2, 3, 1, 4]`

In [14]:
def trace_right(nums):
    n = len(nums); dp = [0] * n; st = []
    for i in range(n - 1, -1, -1):
        eaten = []
        while st and nums[i] > nums[st[-1]]:
            j = st.pop(); dp[i] = max(dp[i] + 1, dp[j]); eaten.append(nums[j])
        st.append(i)
        print(f"i={i} nums={nums[i]:<3} ăn {eaten!s:<12} dp={dp}")
    return max(dp)

print("đáp án:", trace_right([10, 1, 2, 3, 1, 4]), "| brute:", brute([10, 1, 2, 3, 1, 4]))

i=5 nums=4   ăn []           dp=[0, 0, 0, 0, 0, 0]
i=4 nums=1   ăn []           dp=[0, 0, 0, 0, 0, 0]
i=3 nums=3   ăn [1]          dp=[0, 0, 0, 1, 0, 0]
i=2 nums=2   ăn []           dp=[0, 0, 0, 1, 0, 0]
i=1 nums=1   ăn []           dp=[0, 0, 0, 1, 0, 0]
i=0 nums=10  ăn [1, 2, 3, 4] dp=[4, 0, 0, 1, 0, 0]
đáp án: 4 | brute: 4


## 9. Code nộp LeetCode

In [15]:
from typing import List

class Solution:
    def totalSteps(self, nums: List[int]) -> int:
        st = []                                   # (giá trị, bước bị xóa), giá trị giảm ngặt từ đáy lên
        ans = 0
        for x in nums:
            t = 0
            while st and st[-1][0] <= x:          # các phần tử nằm giữa "lớn hơn gần nhất" và x
                t = max(t, st.pop()[1])
            t = t + 1 if st else 0                # không có ai lớn hơn bên trái => không bao giờ bị xóa
            if t > ans:
                ans = t
            st.append((x, t))
        return ans

In [16]:
sol = Solution()
for a, exp in examples:
    got = sol.totalSteps(a)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

3 | kỳ vọng 3 | OK
0 | kỳ vọng 0 | OK


## 10. Vì sao thuật toán đúng

Ký hiệu `L_r(i)` = hàng xóm trái của `i` (nếu còn sống) ở **đầu** bước `r`.

**Bổ đề 1.** Nếu `i` không có phần tử lớn hơn bên trái (`pg[i] = −1`) thì `i` không bao giờ bị xóa. *Chứng minh:* mọi phần tử bên trái `i` đều ≤ `nums[i]`, nên `nums[L_r(i)] > nums[i]` không bao giờ xảy ra.

**Bổ đề 2.** Nếu `j = pg[i] ≠ −1` và `M = max(T[k] : j < k < i)` (0 nếu rỗng), thì `T[i] = M + 1`. *Chứng minh* (quy nạp theo `i`, giả sử mọi `k < i` đã có `T[k]` đúng và các `k` giữa `j` và `i` đều bị xóa — đúng vì `pg[k] ≥ j` nên `pg[k] ≠ −1`):
- Ở bước `r ≤ M`: còn ít nhất một `k ∈ (j, i)` sống ở đầu bước `r` (phần tử có `T[k] = M ≥ r`). Phần tử sống gần `i` nhất trong số đó là `L_r(i)`, và `nums[k] ≤ nums[i]` ⇒ `i` không bị xóa ở bước `r`.
- Ở đầu bước `M + 1`: mọi `k ∈ (j, i)` đã bị xóa, nên hàng xóm trái của `i` là phần tử **còn sống** gần nhất ở vị trí `≤ j`. Nếu `j` còn sống thì đó là `j`, và `nums[j] > nums[i]`. Nếu `j` đã bị xóa thì theo bất biến (B) dưới đây, phần tử đó có giá trị `> nums[j] > nums[i]`. Cả hai trường hợp ⇒ `i` bị xóa ở bước `M + 1`, tức `T[i] = M + 1`.

**Bất biến (B).** Tại mọi thời điểm, với mỗi phần tử `k` đã bị xóa, phần tử còn sống gần nhất bên trái `k` có giá trị `> nums[k]`. *Chứng minh* (quy nạp theo bước): xét bước `r`, gọi `R = r₁, r₂, …` là một đoạn **liên tiếp** (trong mảng hiện tại) các phần tử bị xóa ở bước này, và `g` là phần tử sống ngay trước `R` (không bị xóa ở bước `r`). Vì `r₁` bị xóa, `g > r₁`; vì mỗi `r_{t+1}` bị xóa, `r_t > r_{t+1}` ⇒ `g` lớn hơn mọi phần tử của `R`. Một phần tử `k` đã bị xóa từ trước, nằm giữa `g` và phần tử cuối của `R`, có phần tử sống gần nhất bên trái (trước bước `r`) là `g` hoặc một `r_t`, cả hai đều `≤ g` và `> nums[k]` theo giả thiết quy nạp ⇒ sau bước `r`, phần tử sống gần nhất bên trái của nó là `g`, vẫn `> nums[k]`. ∎

**Định lý.** Số bước = `max T[i]`. Sau bước `max T` không ai bị xóa; trước đó mỗi bước `r ≤ max T` có ít nhất phần tử với `T = r` bị xóa, nên đúng là đã thực hiện `max T` bước.

**Ngăn xếp tính đúng `T`.** Bất biến: trước khi xử lý `i`, ngăn xếp chứa (từ đáy lên) đúng các chỉ số `k < i` mà không có phần tử nào trong `(k, i)` **≥** `nums[k]`, với giá trị giảm ngặt. Vòng `while` pop đúng các phần tử `≤ x`; phần tử còn lại trên đỉnh là `pg[i]`. Mọi `k ∈ (pg[i], i)` hoặc bị pop bây giờ, hoặc đã bị pop bởi `k' ∈ (k, i)` với `T[k'] > T[k]` (vì `k'` có `pg`) — và `k'` cũng thuộc `(pg[i], i)`, lặp lại tới khi gặp phần tử bị pop bây giờ. Vậy `max` trên các phần tử bị pop bây giờ = `max T[k]` trên toàn đoạn. ∎

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng thô | O(n · steps) = O(n²) | O(n) |
| 1b. Công thức `T` trực tiếp | O(n²) | O(n) |
| 2. Ngăn xếp trái → phải | O(n) | O(n) |
| 3. Ngăn xếp phải → trái (doocs) | O(n) | O(n) |
| 4. Danh sách liên kết + ứng viên | O(n log n) (do `sorted` mỗi bước; O(n) nếu bỏ sắp xếp) | O(n) |

In [17]:
rng = random.Random(2289)
N = 10**5
tests = {"ngẫu nhiên": [rng.randint(1, 10**9) for _ in range(N)],
         "[n,1..n-1]": [N] + list(range(1, N)),
         "giảm dần": list(range(N, 0, -1)),
         "răng cưa": [N if i % 1000 == 0 else i % 1000 for i in range(N)]}
for label, arr in tests.items():
    res = []
    for name, f in [("stack_left", stack_left), ("stack_right", stack_right), ("linked_sim", linked_simulation),
                    ("Solution", Solution().totalSteps)]:
        t0 = time.perf_counter(); res.append(f(arr)); dt = time.perf_counter() - t0
        print(f"{label:<11} {name:<11} {dt*1000:7.1f} ms -> {res[-1]}")
    assert len(set(res)) == 1

ngẫu nhiên  stack_left     13.0 ms -> 24


ngẫu nhiên  stack_right    12.4 ms -> 24
ngẫu nhiên  linked_sim     49.4 ms -> 24
ngẫu nhiên  Solution       12.6 ms -> 24
[n,1..n-1]  stack_left     12.2 ms -> 99999
[n,1..n-1]  stack_right    12.5 ms -> 99999


[n,1..n-1]  linked_sim     63.0 ms -> 99999
[n,1..n-1]  Solution       12.7 ms -> 99999
giảm dần    stack_left     18.6 ms -> 1
giảm dần    stack_right     9.0 ms -> 1


giảm dần    linked_sim     28.2 ms -> 1
giảm dần    Solution       34.3 ms -> 1
răng cưa    stack_left     12.1 ms -> 999
răng cưa    stack_right    12.1 ms -> 999
răng cưa    linked_sim     47.7 ms -> 999
răng cưa    Solution       13.5 ms -> 999


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
ALL = {"brute": brute, "quadratic_T": lambda a: max(death_times_quadratic(a)), "stack_left": stack_left,
       "stack_right": stack_right, "linked_simulation": linked_simulation, "Solution": Solution().totalSteps}
edge = examples + [
    ([1], 0),
    ([2, 1], 1),
    ([5, 4, 4], 2),                       # bằng nhau: 4 thứ hai phải chờ 4 thứ nhất
    ([5, 3, 4], 2),                       # bẫy cập nhật tuần tự
    ([3, 2, 1], 1),                       # giảm dần: xóa hết trong 1 bước
    ([10, 1, 2, 3, 4, 5], 5),             # mỗi bước xóa một phần tử
    ([7, 7, 7], 0),
    ([1, 3, 2, 4, 3, 5], 1),
    ([10, 1, 2, 3, 1, 4], 4),
]
for a, exp in edge:
    for name, f in ALL.items():
        assert f(a) == exp, (name, a, f(a), exp)
    print(f"{str(a):<40} -> {exp}")
print("edge OK")

[5, 3, 4, 4, 7, 3, 6, 11, 8, 5, 11]      -> 3
[4, 5, 7, 7, 13]                         -> 0
[1]                                      -> 0
[2, 1]                                   -> 1
[5, 4, 4]                                -> 2
[5, 3, 4]                                -> 2
[3, 2, 1]                                -> 1
[10, 1, 2, 3, 4, 5]                      -> 5
[7, 7, 7]                                -> 0
[1, 3, 2, 4, 3, 5]                       -> 1
[10, 1, 2, 3, 1, 4]                      -> 4
edge OK


### 12.2 Đối chiếu ngẫu nhiên với mô phỏng (seed cố định)

In [19]:
rng = random.Random(42)
for _ in range(3000):
    a = [rng.randint(1, rng.choice([3, 6, 20, 10**9])) for _ in range(rng.randint(1, 14))]
    ref = brute(a)
    for name, f in ALL.items():
        assert f(a) == ref, (name, a)
print("3000 mảng ngẫu nhiên OK")

3000 mảng ngẫu nhiên OK


### 12.3 Tính chất: `T` theo công thức khớp bước bị xóa thật trong mô phỏng; mảng cuối = tiền tố cực đại

In [20]:
def simulate_times(a):
    idx = list(range(len(a))); T = [0] * len(a); step = 0
    while True:
        rm = [idx[k] for k in range(1, len(idx)) if a[idx[k - 1]] > a[idx[k]]]
        if not rm: return T, [a[i] for i in idx]
        step += 1
        for i in rm: T[i] = step
        s = set(rm); idx = [i for i in idx if i not in s]

rng = random.Random(7)
for _ in range(2000):
    a = [rng.randint(1, 6) for _ in range(rng.randint(1, 15))]
    T, final = simulate_times(a)
    assert T == death_times_quadratic(a)
    assert final == prefix_maxima(a)
print("tính chất OK")

tính chất OK


### 12.4 Edge cực trị: đáp án lớn nhất có thể là `n − 1`

In [21]:
N = 10**5
a = [N] + list(range(1, N))
print(Solution().totalSteps(a), "(kỳ vọng", N - 1, ")")
assert Solution().totalSteps(a) == N - 1
assert Solution().totalSteps(list(range(N, 0, -1))) == 1

99999 (kỳ vọng 99999 )


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Mô phỏng **tuần tự** (xóa ngay, so với hàng xóm đã thay đổi trong cùng bước)

In [22]:
def wrong_sequential(nums):
    a = list(nums); steps = 0
    while True:
        b = [a[0]]
        for x in a[1:]:
            if not (b[-1] > x): b.append(x)                 # SAI: b[-1] là phần tử đã qua lọc của bước này
        if len(b) == len(a): return steps
        a = b; steps += 1

for a in [[5, 3, 4], examples[0][0]]:
    print("bản sai :", wrong_sequential(a), "| bản đúng:", Solution().totalSteps(a))

bản sai : 1 | bản đúng: 2
bản sai : 1 | bản đúng: 3


**Vì sao sai:** bản tuần tự xóa luôn mọi phần tử nhỏ hơn "phần tử còn lại gần nhất" ⇒ chỉ cần 1 bước cho mọi mảng không tăng dần. **Sửa:** quyết định xóa dựa trên mảng **đầu bước**.

### Lỗi 2 — Trong ngăn xếp, chỉ pop phần tử **nhỏ hơn ngặt** (`<`) thay vì `≤`

In [23]:
def wrong_strict_pop(nums):
    st = []; ans = 0
    for x in nums:
        t = 0
        while st and st[-1][0] < x:                         # SAI: phần tử bằng x vẫn ở lại
            t = max(t, st.pop()[1])
        t = t + 1 if st else 0
        ans = max(ans, t); st.append((x, t))
    return ans

for a in [[5, 4, 4], examples[0][0]]:
    print("bản sai :", wrong_strict_pop(a), "| bản đúng:", Solution().totalSteps(a))

bản sai : 1 | bản đúng: 2
bản sai : 2 | bản đúng: 3


**Vì sao sai:** phần tử bằng `x` đứng trước `x` **không** xóa được `x` (đề cần `>` ngặt) — nó là phần tử "ở giữa" mà `x` phải chờ. Để nó lại ngăn xếp thì `x` coi nó là "kẻ xóa" và `T = 1`. **Sửa:** pop `≤ x`.

### Lỗi 3 — Cộng 1 cho **mỗi** phần tử bị pop (đếm số phần tử bị ăn thay vì thời gian)

In [24]:
def wrong_count_pops(nums):
    n = len(nums); dp = [0] * n; st = []
    for i in range(n - 1, -1, -1):
        while st and nums[i] > nums[st[-1]]:
            st.pop(); dp[i] += 1                            # SAI: bỏ qua thời gian chờ dp[top]
        st.append(i)
    return max(dp)

for a in [[10, 1, 2, 3, 4, 5], [3, 2, 1, 1, 2]]:
    print("bản sai :", wrong_count_pops(a), "| bản đúng:", Solution().totalSteps(a))

bản sai : 5 | bản đúng: 5
bản sai : 2 | bản đúng: 3


**Vì sao sai:** với `[10, 1, 2, 3, 4, 5]` hai cách **trùng hợp** bằng nhau: `10` xóa lần lượt 5 phần tử, mỗi bước một phần tử, không phải chờ ai. Với `[3, 2, 1, 1, 2]` (phản ví dụ nhỏ nhất, tìm bằng vét cạn): `3` chỉ "ăn" trực tiếp hai phần tử là `2` (chỉ số 1) và `2` (chỉ số 4). Nhưng `2` ở chỉ số 4 phải **chờ** hai số `1` bị xóa trước: số `1` thứ hai chỉ lộ ra sau bước 1, nên `2` cuối bị xóa ở bước 3, không phải bước 2. Bản đếm số lần pop bỏ qua thời gian chờ `dp[top]`, nên luôn cho một **cận dưới** (với cặp `i`→victim, mỗi bước `i` xóa đúng một phần tử) và hụt khi có chờ. **Sửa:** `dp[i] = max(dp[i] + 1, dp[top])`.

### Lỗi 4 — Quên đặt `T = 0` khi ngăn xếp rỗng (luôn `t + 1`)

In [25]:
def wrong_no_reset(nums):
    st = []; ans = 0
    for x in nums:
        t = 0
        while st and st[-1][0] <= x:
            t = max(t, st.pop()[1])
        t = t + 1                                           # SAI: phần tử là max tiền tố không bao giờ bị xóa
        ans = max(ans, t); st.append((x, t))
    return ans

for a in [examples[1][0], [3, 1, 5, 2]]:
    print("bản sai :", wrong_no_reset(a), "| bản đúng:", Solution().totalSteps(a))

bản sai : 5 | bản đúng: 0
bản sai : 2 | bản đúng: 1


**Sửa:** `t = t + 1 if st else 0`.

### Lỗi 5 — Nghĩ đáp án là **độ dài đoạn giảm dài nhất** hoặc **số vị trí giảm**

In [26]:
def wrong_descents(nums):
    return sum(1 for i in range(1, len(nums)) if nums[i - 1] > nums[i])

def wrong_longest_run(nums):
    best = cur = 0
    for i in range(1, len(nums)):
        cur = cur + 1 if nums[i - 1] > nums[i] else 0
        best = max(best, cur)
    return best

a = examples[0][0]
print("số vị trí giảm:", wrong_descents(a), "| đoạn giảm dài nhất:", wrong_longest_run(a), "| bản đúng:", Solution().totalSteps(a))

số vị trí giảm: 4 | đoạn giảm dài nhất: 2 | bản đúng: 3


**Vì sao sai:** bước sau phụ thuộc vào cấu trúc **sau khi xóa** (các phần tử "lộ" ra), không suy ra được từ thống kê cục bộ của mảng ban đầu.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Quá trình "xóa đồng thời theo hàng xóm" ⇒ tính **thời điểm bị xóa** của từng phần tử; đáp án thường là `max`.
- "Ai xóa tôi?" thường là **phần tử lớn hơn / nhỏ hơn gần nhất** ⇒ ngăn xếp đơn điệu.
- Khi pop, **gộp thông tin** (max thời gian) của những phần tử bị pop vào phần tử mới.
- Cẩn thận dấu `<` / `≤` khi có phần tử **bằng nhau**: xác định rõ "bằng" có xóa được hay không.
- Luôn viết mô phỏng O(n²) để đối chiếu — các bài "process" rất dễ sai một đơn vị.
- Bài cùng họ: 735 Asteroid Collision, 1944 Number of Visible People in a Queue, 2282 Number of People That Can Be Seen in a Grid, 1475 Final Prices With a Special Discount, 2832 Maximal Range That Each Element Is Maximum in It.

## 15. Biến thể của bài

### 15.1 Biến thể A — Trả về **bước bị xóa của từng phần tử** (và mảng cuối)

In [27]:
def death_times(nums):
    st = []; T = [0] * len(nums)
    for i, x in enumerate(nums):
        t = 0
        while st and nums[st[-1]] <= x:
            t = max(t, T[st.pop()])
        T[i] = t + 1 if st else 0
        st.append(i)
    return T

rng = random.Random(151)
for _ in range(1000):
    a = [rng.randint(1, 6) for _ in range(rng.randint(1, 15))]
    assert death_times(a) == simulate_times(a)[0]
print(death_times(examples[0][0]))

[0, 1, 2, 3, 0, 1, 2, 0, 1, 1, 0]


### 15.2 Biến thể B — Xóa khi `nums[i − 1] ≥ nums[i]` (cả bằng nhau cũng bị xóa) — cho tới khi **tăng ngặt**

In [28]:
def steps_strict(nums):
    st = []; ans = 0
    for x in nums:
        t = 0
        while st and st[-1][0] < x:                         # chỉ phần tử < x mới là "ở giữa"
            t = max(t, st.pop()[1])
        t = t + 1 if st else 0                              # đỉnh còn lại >= x sẽ xóa x
        ans = max(ans, t); st.append((x, t))
    return ans

def brute_strict(nums):
    a = list(nums); s = 0
    while True:
        b = [a[0]] + [a[i] for i in range(1, len(a)) if not (a[i - 1] >= a[i])]
        if len(b) == len(a): return s
        a = b; s += 1

rng = random.Random(152)
for _ in range(2000):
    a = [rng.randint(1, 5) for _ in range(rng.randint(1, 12))]
    assert steps_strict(a) == brute_strict(a)
print(steps_strict([5, 4, 4]), steps_strict([7, 7, 7]), "(so với bản gốc:", Solution().totalSteps([5, 4, 4]), Solution().totalSteps([7, 7, 7]), ")")

1 1 (so với bản gốc: 2 0 )


Thú vị: "Lỗi 2" ở mục 13 chính là **lời giải đúng** của biến thể này — dấu so sánh quyết định bài toán.

### 15.3 Biến thể C — Xóa theo hàng xóm **phải** (`nums[i] > nums[i + 1]` ⇒ xóa `nums[i]`)

Đối xứng: đảo mảng và đổi chiều so sánh (phần tử bị xóa nếu **lớn hơn** hàng xóm phải ⇔ trong mảng đảo, lớn hơn hàng xóm trái ⇔ dùng giá trị âm).

In [29]:
def steps_right_neighbor(nums):
    return Solution().totalSteps([-x for x in reversed(nums)])

def brute_right(nums):
    a = list(nums); s = 0
    while True:
        b = [a[i] for i in range(len(a) - 1) if not (a[i] > a[i + 1])] + [a[-1]]
        if len(b) == len(a): return s
        a = b; s += 1

rng = random.Random(153)
for _ in range(2000):
    a = [rng.randint(1, 6) for _ in range(rng.randint(1, 12))]
    assert steps_right_neighbor(a) == brute_right(a)
print(steps_right_neighbor([1, 2, 3, 0]), brute_right([1, 2, 3, 0]))

3 3


### 15.4 Biến thể D — Tổng số phần tử bị xóa (dễ: `n − số tiền tố cực đại`)

In [30]:
def removed_count(nums):
    return len(nums) - len(prefix_maxima(nums))
print(removed_count(examples[0][0]), "= 11 - 4")

7 = 11 - 4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | định nghĩa | O(n²) | O(n) | kiểm chứng |
| 1b. Công thức `T` trực tiếp | pg + max đoạn | O(n²) | O(n) | hiểu bản chất |
| 2. Ngăn xếp trái → phải | gộp max khi pop | O(n) | O(n) | **mặc định**, dễ chứng minh |
| 3. Ngăn xếp phải → trái | `dp = max(dp+1, dp[top])` | O(n) | O(n) | cách doocs, ngắn |
| 4. Danh sách liên kết | chỉ xét ứng viên | O(n) ~ O(n log n) | O(n) | cần mảng/thứ tự xóa thật |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `totalSteps([6, 5, 4, 3, 2, 1])` và `totalSteps([1, 9, 2, 3, 4])`?

**Bài 2 (tính tay):** Tính mảng `T` cho `[8, 2, 5, 3, 9, 1, 1]` và đáp án.

**Bài 3 (điền TODO):** Viết `bt3_todo(nums)` trả về **danh sách các phần tử bị xóa ở từng bước** (danh sách các danh sách), chạy O(n) dựa trên `T`.

**Bài 4 (tự viết lại):** Viết `bt4_todo(nums)` cài cách ngăn xếp phải → trái không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(nums)` trả về số bước khi mỗi bước xóa mọi phần tử **nhỏ hơn cả hai hàng xóm** (phần tử đầu/cuối chỉ có một hàng xóm thì không bao giờ bị xóa). Viết bản mô phỏng đúng định nghĩa là đủ.

In [31]:
def bt3_todo(nums):
    # TODO: các phần tử bị xóa ở từng bước
    pass

def bt4_todo(nums):
    # TODO: ngăn xếp phải -> trái
    pass

def bt5_todo(nums):
    # TODO: xóa phần tử nhỏ hơn cả hai hàng xóm
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: giảm dần ⇒ mọi phần tử trừ đầu bị xóa ngay. Mức 2: `[1,9,2,3,4]`: `pg` của `2, 3, 4` đều là `9`. Mức 3: `1` và `3`.

**Bài 2.** Mức 1: `pg = [−, 0, 0, 2, −, 4, 4]`. Mức 2: `T[1] = 1`, `T[2] = 1 + T[1] = 2`, `T[3] = 1`. Mức 3: `T[5] = 1`, `T[6] = 1 + T[5] = 2` ⇒ `T = [0,1,2,1,0,1,2]`, đáp án `2`.

**Bài 3.** Mức 1: tính `T` bằng ngăn xếp. Mức 2: tạo `max(T)` danh sách rỗng. Mức 3: duyệt `i` tăng dần, `out[T[i] − 1].append(nums[i])` nếu `T[i] > 0`.

**Bài 4.** Mức 1: duyệt `i` từ phải. Mức 2: `while st and nums[i] > nums[st[-1]]`. Mức 3: `dp[i] = max(dp[i] + 1, dp[st.pop()])`, trả `max(dp)`.

**Bài 5.** Mức 1: quyết định theo mảng đầu bước. Mức 2: `i` bị xóa nếu `0 < i < len−1` và `a[i] < a[i−1]` và `a[i] < a[i+1]`. Mức 3: lặp tới khi không xóa được.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `1`, `3`. **Bài 2:** `T = [0,1,2,1,0,1,2]`, đáp án `2`.

In [32]:
assert Solution().totalSteps([6, 5, 4, 3, 2, 1]) == 1
assert Solution().totalSteps([1, 9, 2, 3, 4]) == 3
assert death_times([8, 2, 5, 3, 9, 1, 1]) == [0, 1, 2, 1, 0, 1, 2]
assert Solution().totalSteps([8, 2, 5, 3, 9, 1, 1]) == 2

def bt3(nums):
    T = death_times(nums); out = [[] for _ in range(max(T, default=0))]
    for x, t in zip(nums, T):
        if t: out[t - 1].append(x)
    return out

def bt4(nums):
    return stack_right(nums)

def bt5(nums):
    a = list(nums); s = 0
    while True:
        keep = [a[i] for i in range(len(a)) if not (0 < i < len(a) - 1 and a[i] < a[i - 1] and a[i] < a[i + 1])]
        if len(keep) == len(a): return s
        a = keep; s += 1

def removed_by_step_brute(nums):
    a = list(nums); out = []
    while True:
        rm = [a[i] for i in range(1, len(a)) if a[i - 1] > a[i]]
        if not rm: return out
        out.append(rm); a = one_step(a)

rng = random.Random(1987)
for _ in range(2000):
    a = [rng.randint(1, 6) for _ in range(rng.randint(1, 14))]
    assert bt3(a) == removed_by_step_brute(a)
    assert bt4(a) == brute(a)
print(bt3(examples[0][0]), bt5([5, 1, 4, 2, 6, 3, 7]), bt5([3, 1, 2]))
print("Đáp án bài tập: tất cả assert OK")

[[3, 3, 8, 5], [4, 6], [4]] 2 1
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
st, ans = [], 0                      # (giá trị, bước bị xóa), giá trị giảm ngặt
for x in nums:
    t = 0
    while st and st[-1][0] <= x:     # các phần tử "ở giữa" (<= x)
        t = max(t, st.pop()[1])
    t = t + 1 if st else 0           # có kẻ lớn hơn bên trái => bị xóa sau khi phần giữa hết
    ans = max(ans, t); st.append((x, t))
return ans
```

**Checklist:**
- [ ] Xóa **đồng thời** theo trạng thái đầu bước (khi mô phỏng).
- [ ] Pop `≤ x` (phần tử bằng `x` không xóa được `x`).
- [ ] Ngăn xếp rỗng ⇒ `T = 0`.
- [ ] Gộp **max** thời gian khi pop, không cộng dồn số lần pop.